In [4]:
import pandas as pd

In [33]:
task = "https://github.com/PleschNA/testGA/raw/main/Task1.xlsx"
df = pd.read_excel(task)
print(df.head())

     Бренд   Код товара                                Наименование товара  \
0  Clarins  19000412437        CLARINS Тушь д/ресниц Wonder volume xxl 8мл   
1  Clarins  14037100013  Clarins Флюид тональный устойчивый с омолажива...   
2  Clarins  19000033387  CLARINS Масло-блеск для губ Lip Comfort Oil т....   
3  Clarins  14037100015  Clarins Флюид тональный устойчивый с омолажива...   
4  Clarins  19000168458       Clarins Масло-блеск д/губ Lip Comfort 17 7мл   

  Категория1 Категория2            Категория3               Категория4  \
0     Макияж      Глаза       Тушь для ресниц                      NaN   
1     Макияж       Лицо      Тональные основы  Жидкие тональные основы   
2     Макияж       Губы  Масло для губ_макияж                      NaN   
3     Макияж       Лицо      Тональные основы  Жидкие тональные основы   
4     Макияж       Губы  Масло для губ_макияж                      NaN   

  Сегмент  Цена закупки товара  letu.ru  iledebeaute.ru  rivegauche.ru  \
0    Люкс   

In [35]:
markup = "https://github.com/PleschNA/testGA/raw/main/markup.xlsx"
mu = pd.read_excel(markup)
print(mu.head())

  Сегмент рынка Категория Бренд  Продукт Дата начала Дата окончания  Наценка  \
0          Люкс       NaN   NaN      NaN  2025-01-01            NaT      1.2   
1    Массмаркет       NaN   NaN      NaN  2025-01-01            NaT      0.7   
2          Люкс    Макияж   NaN      NaN  2025-01-01            NaT      0.9   
3          Люкс      Уход   NaN      NaN  2025-01-01            NaT      1.5   
4    Массмаркет    Макияж   NaN      NaN  2025-01-01            NaT      0.5   

  Активный  
0       Да  
1       Да  
2       Да  
3       Да  
4       Да  


In [36]:
print(df.columns.tolist())
print(df.info())
print(df.shape)

['Бренд', 'Код товара', 'Наименование товара', 'Категория1', 'Категория2', 'Категория3', 'Категория4', 'Сегмент', 'Цена закупки товара', 'letu.ru', 'iledebeaute.ru', 'rivegauche.ru', 'Ozon', 'Wildberries', 'MagnitMarket', 'Megamarket', 'lamoda.ru', 'tsum.ru', 'visagehall.ru']
<class 'pandas.DataFrame'>
RangeIndex: 683 entries, 0 to 682
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Бренд                683 non-null    str    
 1   Код товара           683 non-null    int64  
 2   Наименование товара  683 non-null    str    
 3   Категория1           683 non-null    str    
 4   Категория2           683 non-null    str    
 5   Категория3           683 non-null    str    
 6   Категория4           297 non-null    str    
 7   Сегмент              683 non-null    str    
 8   Цена закупки товара  683 non-null    int64  
 9   letu.ru              193 non-null    float64
 10  iledebeaute.ru       495

In [37]:
# часть 1 
print("Товаров:", len(df))
print("Правил наценки:", len(mu))
print(mu.columns.tolist())

Товаров: 683
Правил наценки: 123
['Сегмент рынка', 'Категория', 'Бренд', 'Продукт', 'Дата начала', 'Дата окончания', 'Наценка', 'Активный']


In [38]:
today = pd.Timestamp("2025-10-01")

mu["Дата начала"]    = pd.to_datetime(mu["Дата начала"])
mu["Дата окончания"] = pd.to_datetime(mu["Дата окончания"])

active = mu[
    (mu["Активный"] == "Да") &
    (mu["Дата начала"] <= today) &
    ((mu["Дата окончания"].isna()) | (mu["Дата окончания"] >= today))
]

print("Активных правил:", len(active))
print(active["Активный"].value_counts())

Активных правил: 120
Активный
Да    120
Name: count, dtype: int64


In [45]:
def find_markup(row):
    brend   = row["Бренд"]
    kat1    = row["Категория1"]
    segment = row["Сегмент"]

    # Код товара без ".0"
    product = str(row["Код товара"]).split(".")[0]
    # Продукт в справочнике тоже без ".0"
    mu_product = active["Продукт"].astype(str).str.split(".").str[0]

    # 1. Продукт
    m = active[mu_product == product]
    if not m.empty:
        return m.iloc[0]["Наценка"], "Продукт"

    # 4. Бренд + Категория1
    m = active[(active["Бренд"] == brend) & (active["Категория"] == kat1)]
    if not m.empty:
        return m.iloc[0]["Наценка"], "Бренд/Кат1"

    # 5. Только Бренд
    m = active[(active["Бренд"] == brend) & (active["Категория"].isna())]
    if not m.empty:
        return m.iloc[0]["Наценка"], "Бренд"

    # 8. Сегмент + Категория1
    m = active[(active["Сегмент рынка"] == segment) & (active["Категория"] == kat1)]
    if not m.empty:
        return m.iloc[0]["Наценка"], "Сегмент/Кат1"

    # 9. Только Сегмент
    m = active[(active["Сегмент рынка"] == segment) & (active["Категория"].isna())]
    if not m.empty:
        return m.iloc[0]["Наценка"], "Сегмент"

    return None, "Не найдено"

In [46]:
df[["Наценка", "Уровень_правила"]] = df.apply(
    lambda r: pd.Series(find_markup(r)), axis=1
)

df["Цена_до_скидки"] = df["Цена закупки товара"] * df["Наценка"]

print("Наценки найдены для:", df["Наценка"].notna().sum(), "из", len(df))
print()
print(df["Уровень_правила"].value_counts())

Наценки найдены для: 683 из 683

Уровень_правила
Бренд      679
Продукт      4
Name: count, dtype: int64


In [47]:
# Какие продукты есть в справочнике
print("Продукты в mu:")
print(active[active["Продукт"].notna()][["Продукт", "Наценка", "Активный"]])

# Тип данных
print("\nТип Продукт в mu:", active["Продукт"].dtype)

# Есть ли эти коды в df?
product_codes = active[active["Продукт"].notna()]["Продукт"].astype(str).tolist()
print("\nИщем коды:", product_codes)
print("Есть в df:", df[df["Код товара"].astype(str).isin(product_codes)][["Код товара", "Наименование товара"]])

Продукты в mu:
          Продукт  Наценка Активный
119  1.900022e+10      2.5       Да
120  1.900042e+10      2.5       Да
121  1.900036e+10      2.5       Да
122  1.403590e+10      2.0       Да

Тип Продукт в mu: float64

Ищем коды: ['19000217070.0', '19000419067.0', '19000359936.0', '14035900003.0']
Есть в df: Empty DataFrame
Columns: [Код товара, Наименование товара]
Index: []


In [48]:
#Часть 2
competitors = [
    "letu.ru", "iledebeaute.ru", "rivegauche.ru", "Ozon",
    "Wildberries", "MagnitMarket", "Megamarket",
    "lamoda.ru", "tsum.ru", "visagehall.ru"
]
print("Конкурентов:", len(competitors))

Конкурентов: 10


In [49]:
for c in competitors:
    df[f"Индекс_{c}"] = df["Цена_до_скидки"] / df[c] * 100

In [52]:
df["Мин_у_конкурентов"] = df[competitors].min(axis=1)
df["Индекс_к_минимуму"] = df["Цена_до_скидки"] / df["Мин_у_конкурентов"] * 100

In [53]:
summary = []
for c in competitors:
    idx = df[f"Индекс_{c}"].dropna()
    summary.append({
        "Конкурент": c,
        "Позиций с ценой": len(idx),
        "Средний индекс, %": round(idx.mean(), 1),
        "Медиана, %": round(idx.median(), 1),
        "Мы дороже, % позиций": round((idx > 100).mean() * 100, 1),
    })

summary_df = pd.DataFrame(summary).sort_values("Средний индекс, %", ascending=False)
print(summary_df.to_string(index=False))

     Конкурент  Позиций с ценой  Средний индекс, %  Медиана, %  Мы дороже, % позиций
 visagehall.ru              388               61.0        60.2                   0.8
          Ozon              471               59.4        59.8                   1.1
iledebeaute.ru              495               58.3        58.5                   3.8
     lamoda.ru              364               58.0        58.8                   0.5
 rivegauche.ru              610               57.3        59.0                   0.5
       tsum.ru              226               51.6        51.1                   0.4
   Wildberries              176               47.6        48.0                   0.0
  MagnitMarket              120               44.9        41.8                   0.0
    Megamarket               89               43.9        40.3                   0.0
       letu.ru              193               43.6        40.8                   0.0


In [54]:
# Проверка: сравним закупку, нашу цену и цену конкурентов для нескольких товаров
sample = df[df["Мин_у_конкурентов"].notna()].head(15)[
    ["Бренд", "Наименование товара", "Цена закупки товара",
     "Наценка", "Цена_до_скидки",
     "Мин_у_конкурентов", "Индекс_к_минимуму"]
]
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.width", 200)
print(sample.to_string())

      Бренд                                                                                  Наименование товара  Цена закупки товара  Наценка  Цена_до_скидки  Мин_у_конкурентов  Индекс_к_минимуму
0   Clarins                                                          CLARINS Тушь д/ресниц Wonder volume xxl 8мл                  983     1.03         1012.49             1029.0          98.395530
1   Clarins                             Clarins Флюид тональный устойчивый с омолаживающим действием 30мл 108.5W                 1957     1.03         2015.71             2107.0          95.667299
2   Clarins                                                CLARINS Масло-блеск для губ Lip Comfort Oil т.10 7 мл                  850     1.03          875.50              919.0          95.266594
3   Clarins  Clarins Флюид тональный устойчивый с омолаживающим действием SPF15 Everlasting Youth Fluid 110 30мл                 2145     1.03         2209.35             2355.0          93.815287
4   Clarins    

In [55]:
sample_vs = df[
    (df["Бренд"] == "Vivienne Sabo") & 
    (df["Мин_у_конкурентов"].notna())
].head(10)[
    ["Наименование товара", "Цена закупки товара", "Наценка", 
     "Цена_до_скидки", "Мин_у_конкурентов", "Индекс_к_минимуму"]
].round(2)

print(sample_vs.to_string(index=False))

                                                         Наименование товара  Цена закупки товара  Наценка  Цена_до_скидки  Мин_у_конкурентов  Индекс_к_минимуму
                            Vivienne Sabo Блеск д/губ Le grand Volume 10 3мл                  258     0.59          152.22              259.0              58.77
                Vivienne Sabo Тональный крем д/лица Cabaret Premiere 06 25мл                  281     0.59          165.79              283.0              58.58
                Vivienne Sabo Тональный крем д/лица Cabaret Premiere 03 25мл                  368     0.59          217.12              371.0              58.52
Vivienne Sabo Набор 2024: тушь Cabaret Premiere 01+гель д/бровей Fixateur 02                  548     0.59          323.32              554.0              58.36
                               Vivienne Sabo Тени д/век Hypnostique 04 0,6гр                  341     0.59          201.19              345.0              58.32
                     Vivienne Sabo

In [56]:
# Гипотеза: 0.59 — коэффициент к минимальной цене конкурентов
df["Цена_гипотеза"] = df["Мин_у_конкурентов"] * 0.59
# Сравнить с закупкой — не ниже ли?
df["Маржа_гипотеза"] = (df["Цена_гипотеза"] - df["Цена закупки товара"]) / df["Цена_гипотеза"]
# Посмотреть на Vivienne Sabo
print(df[df["Бренд"]=="Vivienne Sabo"][["Цена закупки товара", "Мин_у_конкурентов",
                                          "Цена_гипотеза", "Маржа_гипотеза"]].head(10))

    Цена закупки товара  Мин_у_конкурентов  Цена_гипотеза  Маржа_гипотеза
35                  258              259.0         152.81       -0.688371
36                  281              283.0         166.97       -0.682937
37                  368              371.0         218.89       -0.681210
38                  548              554.0         326.86       -0.676559
39                  341              345.0         203.55       -0.675264
40                  242              245.0         144.55       -0.674161
41                  153              155.0          91.45       -0.673045
43                  187              190.0         112.10       -0.668153
44                  190              194.0         114.46       -0.659969
45                  128              131.0          77.29       -0.656100


In [57]:
def apply_markup(zakupka, nacenka):
    if nacenka >= 1:
        return zakupka * nacenka          # множитель (Clarins: 1.03 → ×1.03)
    else:
        return zakupka * (1 + nacenka)    # процент (Vivienne Sabo: 0.59 → ×1.59)

In [58]:
# Применяем универсальную формулу наценки
df["Цена_до_скидки"] = df.apply(
    lambda r: apply_markup(r["Цена закупки товара"], r["Наценка"]), axis=1
)

# Проверим итог
print(df[["Бренд", "Цена закупки товара", "Наценка", "Цена_до_скидки"]].head(10))
print()
print("Средняя цена до скидки по брендам:")
print(df.groupby("Бренд")["Цена_до_скидки"].mean().round(2))
print()
print("Маржа по брендам (от цены продажи):")
df["Маржа_%"] = (df["Цена_до_скидки"] - df["Цена закупки товара"]) / df["Цена_до_скидки"] * 100
print(df.groupby("Бренд")["Маржа_%"].mean().round(1))

     Бренд  Цена закупки товара  Наценка  Цена_до_скидки
0  Clarins                  983     1.03         1012.49
1  Clarins                 1957     1.03         2015.71
2  Clarins                  850     1.03          875.50
3  Clarins                 2145     1.03         2209.35
4  Clarins                  835     1.03          860.05
5  Clarins                  898     1.03          924.94
6  Clarins                  983     1.03         1012.49
7  Clarins                 1340     1.03         1380.20
8  Clarins                 3458     1.03         3561.74
9  Clarins                  961     1.03          989.83

Средняя цена до скидки по брендам:
Бренд
Clarins          2046.76
Vivienne Sabo     461.34
Name: Цена_до_скидки, dtype: float64

Маржа по брендам (от цены продажи):
Бренд
Clarins           3.4
Vivienne Sabo    37.1
Name: Маржа_%, dtype: float64


In [59]:
# Заново считаем индекс по каждому конкуренту
for c in competitors:
    df[f"Индекс_{c}"] = df["Цена_до_скидки"] / df[c] * 100

# Минимум у конкурентов и наш индекс к минимуму
df["Мин_у_конкурентов"] = df[competitors].min(axis=1)
df["Индекс_к_минимуму"] = df["Цена_до_скидки"] / df["Мин_у_конкурентов"] * 100

# Сводная таблица
summary = []
for c in competitors:
    idx = df[f"Индекс_{c}"].dropna()
    summary.append({
        "Конкурент": c,
        "Позиций с ценой": len(idx),
        "Средний индекс, %": round(idx.mean(), 1),
        "Медиана, %": round(idx.median(), 1),
        "Мы дороже, % позиций": round((idx > 100).mean() * 100, 1),
    })

summary_df = pd.DataFrame(summary).sort_values("Средний индекс, %", ascending=False)
print(summary_df.to_string(index=False))

     Конкурент  Позиций с ценой  Средний индекс, %  Медиана, %  Мы дороже, % позиций
   Wildberries              176              128.1       129.3                  94.9
  MagnitMarket              120              120.9       112.7                 100.0
    Megamarket               89              118.3       108.5                  58.4
       letu.ru              193              117.5       109.9                  96.4
          Ozon              471               95.5        65.0                  35.5
iledebeaute.ru              495               89.9        64.4                  37.8
 rivegauche.ru              610               81.8        63.3                  33.3
     lamoda.ru              364               80.0        62.8                  21.4
 visagehall.ru              388               70.5        61.8                  13.4
       tsum.ru              226               51.6        51.1                   0.4


In [60]:
# Минимально допустимая цена при марже 10% (маржа считается от цены продажи)
df["Мин_цена_маржа10"] = df["Цена закупки товара"] / 0.9

# Можем ли мы встать в минимум рынка (с небольшим запасом −1%)?
df["Целевая_цена"] = df["Мин_у_конкурентов"] * 0.99

# Условие: цена ≥ мин.цена для маржи 10%
df["Можно_конкурировать"] = df["Целевая_цена"] >= df["Мин_цена_маржа10"]

# Какая скидка нужна от текущей цены до скидки
df["Нужная_скидка_%"] = (1 - df["Целевая_цена"] / df["Цена_до_скидки"]) * 100
df.loc[~df["Можно_конкурировать"], "Нужная_скидка_%"] = None

# Итоговая рекомендация
def recommend(row):
    if pd.isna(row["Мин_у_конкурентов"]):
        return "Нет данных у конкурентов"
    if row["Цена_до_скидки"] <= row["Мин_у_конкурентов"]:
        return "Уже минимальная цена"
    if row["Можно_конкурировать"]:
        return f"Скидка {row['Нужная_скидка_%']:.0f}%"
    return "Нельзя — маржа < 10%"

df["Рекомендация"] = df.apply(recommend, axis=1)

# Сводка
print(df["Рекомендация"].apply(lambda x: x.split()[0]).value_counts())
print()
print(df[["Бренд", "Наименование товара", "Цена закупки товара",
          "Цена_до_скидки", "Мин_у_конкурентов",
          "Мин_цена_маржа10", "Рекомендация"]].head(15))

Рекомендация
Уже       452
Нельзя    160
Скидка     71
Name: count, dtype: int64

      Бренд                      Наименование товара  Цена закупки товара  Цена_до_скидки  Мин_у_конкурентов  Мин_цена_маржа10          Рекомендация
0   Clarins  CLARINS Тушь д/ресниц Wonder volume ...                  983         1012.49             1029.0       1092.222222  Уже минимальная цена
1   Clarins  Clarins Флюид тональный устойчивый с...                 1957         2015.71             2107.0       2174.444444  Уже минимальная цена
2   Clarins  CLARINS Масло-блеск для губ Lip Comf...                  850          875.50              919.0        944.444444  Уже минимальная цена
3   Clarins  Clarins Флюид тональный устойчивый с...                 2145         2209.35             2355.0       2383.333333  Уже минимальная цена
4   Clarins  Clarins Масло-блеск д/губ Lip Comfor...                  835          860.05              924.0        927.777778  Уже минимальная цена
5   Clarins         CLAR

In [61]:
# Разбивка рекомендаций по брендам
print("Рекомендации × Бренд:")
print(pd.crosstab(df["Бренд"], df["Рекомендация"].apply(lambda x: x.split()[0])))

Рекомендации × Бренд:
Рекомендация   Нельзя  Скидка  Уже
Бренд                             
Clarins             0       4  438
Vivienne Sabo     160      67   14


In [62]:
print("Средняя нужная скидка:")
print(df[df["Рекомендация"].str.startswith("Скидка", na=False)]["Нужная_скидка_%"].describe().round(1))

Средняя нужная скидка:
count    71.0
mean     19.5
std       8.0
min       7.8
25%      14.5
50%      17.8
75%      25.6
max      46.2
Name: Нужная_скидка_%, dtype: float64


In [63]:
# Кто из конкурентов даёт минимальную цену в группе "Нельзя"
blocked = df[df["Рекомендация"] == "Нельзя — маржа < 10%"].copy()

def cheapest_competitor(row):
    prices = row[competitors]
    if prices.isna().all():
        return None
    return prices.idxmin()

blocked["Самый_дешёвый_конкурент"] = blocked.apply(cheapest_competitor, axis=1)
print("Самый дешёвый конкурент для 160 позиций 'Нельзя':")
print(blocked["Самый_дешёвый_конкурент"].value_counts())

Самый дешёвый конкурент для 160 позиций 'Нельзя':
Самый_дешёвый_конкурент
Ozon              87
Wildberries       16
iledebeaute.ru    16
letu.ru           11
Megamarket        10
lamoda.ru          8
rivegauche.ru      6
MagnitMarket       4
visagehall.ru      2
Name: count, dtype: int64


In [64]:
# Для каждой позиции посчитаем, какая наценка нужна, чтобы встать в минимальную цену
# с маржой 10%

# Минимальная цена рынка с −1%
df["Целевая_цена_рынка"] = df["Мин_у_конкурентов"] * 0.99

# Нужная наценка (универсальная формула, обратная apply_markup):
# Если хочется цена > закупки в 1.1+ раз → наценка = цена / закупка
# Но у нас разные шкалы, поэтому просто посмотрим множитель к закупке
df["Нужный_множитель"] = df["Целевая_цена_рынка"] / df["Цена закупки товара"]

# Группируем по бренду и категории — чтобы понять, где какую наценку ставить
grouped = df.groupby(["Бренд", "Категория1"]).agg(
    позиций=("Нужный_множитель", "count"),
    медиана_множителя=("Нужный_множитель", "median"),
    мин_множитель=("Нужный_множитель", "min"),
    макс_множитель=("Нужный_множитель", "max"),
).round(2)

print(grouped)

                                позиций  медиана_множителя  мин_множитель  макс_множитель
Бренд         Категория1                                                                 
Clarins       Макияж                235               1.61           1.04            1.81
              Уход                  206               1.62           1.20            2.01
              Уход за волосами        1               1.68           1.68            1.68
Vivienne Sabo Макияж                238               1.03           0.24            3.07
              Уход                    3               0.84           0.83            2.49


In [66]:
# Предлагаемые наценки
proposals = pd.DataFrame([
    {"Бренд": "Clarins",         "Категория1": "Макияж",         "Текущая": 1.03, "Новая": 1.55},
    {"Бренд": "Clarins",         "Категория1": "Уход",           "Текущая": 1.03, "Новая": 1.55},
    {"Бренд": "Clarins",         "Категория1": "Уход за волосами","Текущая": 1.03, "Новая": 1.60},
    {"Бренд": "Vivienne Sabo",   "Категория1": "Макияж",         "Текущая": 1.59, "Новая": 1.12},
    {"Бренд": "Vivienne Sabo",   "Категория1": "Уход",           "Текущая": 1.59, "Новая": 1.12},
])

proposals["Маржа при новой, %"] = ((proposals["Новая"] - 1) / proposals["Новая"] * 100).round(1)
print(proposals.to_string(index=False))

        Бренд       Категория1  Текущая  Новая  Маржа при новой, %
      Clarins           Макияж     1.03   1.55                35.5
      Clarins             Уход     1.03   1.55                35.5
      Clarins Уход за волосами     1.03   1.60                37.5
Vivienne Sabo           Макияж     1.59   1.12                10.7
Vivienne Sabo             Уход     1.59   1.12                10.7


In [67]:
# Что будет, если применить новые наценки
def new_price(row):
    for _, p in proposals.iterrows():
        if p["Бренд"] == row["Бренд"] and p["Категория1"] == row["Категория1"]:
            return row["Цена закупки товара"] * p["Новая"]
    return row["Цена_до_скидки"]

df["Цена_новая"] = df.apply(new_price, axis=1)
df["Маржа_новая_%"] = (df["Цена_новая"] - df["Цена закупки товара"]) / df["Цена_новая"] * 100

# Сводка по брендам
effect = df.groupby("Бренд").agg(
    позиций=("Цена_новая", "count"),
    средняя_цена_было=("Цена_до_скидки", "mean"),
    средняя_цена_стало=("Цена_новая", "mean"),
    средняя_маржа_было=("Маржа_%", "mean"),
    средняя_маржа_стало=("Маржа_новая_%", "mean"),
).round(2)

print(effect)

               позиций  средняя_цена_было  средняя_цена_стало  средняя_маржа_было  средняя_маржа_стало
Бренд                                                                                                 
Clarins            442            2046.76             3049.04                3.41                35.49
Vivienne Sabo      241             461.34              324.97               37.11                10.71
